In [1]:
import pandas as pd
import sys
sys.path.append('../../')
from MEGA_utilities import data_col_standardize, data_remove_duplicate


# load data
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
origin_df = Crick_H3N2.copy()

## DNA MEGA

In [2]:
## select required columns
H3N2_data_filt1 = origin_df[['serumName','virusName','serumHA', 'serumNA', 'virusHA', 'virusNA', 
                           'serumPassCat','virusPassCat', 'serumType','HI_Dist']].copy()
## remove duplicated row and mean HI_Dist
H3N2_data_filt2 = H3N2_data_filt1.groupby(['serumHA', 'serumNA', 'virusHA', 'virusNA', 'serumPassCat', 'virusPassCat']) \
        .agg({'serumName': 'first', 'virusName': 'first', 'serumType': 'first', 'HI_Dist': 'mean'}) \
        .reset_index()[['serumName', 'virusName', 'serumHA', 'serumNA', 'virusHA', 'virusNA',
                        'serumPassCat', 'virusPassCat', 'serumType', 'HI_Dist']]
## remove PassCat = 'BOTH'
H3N2_data_filt3 = H3N2_data_filt2[(H3N2_data_filt2['serumPassCat'] != 'BOTH') &
                                  (H3N2_data_filt2['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
H3N2_data_filt4 = H3N2_data_filt3.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                       'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})

H3N2_data_final = H3N2_data_filt4.copy()

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

class AADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['serumHA'] + '<eos>' + DataFrame['serumNA'] + '<eos>' + DataFrame['virusHA'] + \
                         '<eos>' + DataFrame['virusNA'] + '<eos>' + DataFrame['serumType'] + \
                         '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat']).tolist()
        self.labels = torch.tensor(DataFrame['HI_Dist'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(H3N2_data_final, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=1/9, random_state=42)

train_dataset = AADataset(train_df)
valid_dataset = AADataset(valid_df)
test_dataset = AADataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [ ]:
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from torch.optim import AdamW
from transformers import get_scheduler
import torch

# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')

# update the num_vocab and num_label
config = MegaConfig() 
config.num_labels=1
config.vocab_size=28
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=3
device = torch.device("cuda:0")
model = MegaForSequenceClassification(config)
model.to(device)
print("Number of parameters: %e"%count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=1e-4)

# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                             num_warmup_steps=len(train_loader), num_training_steps=num_training_steps)

Number of parameters: 6.893510e+05


In [10]:
from tqdm import tqdm
from utilities import print_exams
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from utilities import EarlyStopping
from datetime import datetime

progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=10, delta=0.005, save_dir='../../../trained_model/1.3_H3N2_only_model/')

# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    with torch.no_grad():
        model.eval()
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
            batch_input = batch_input.to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls += logits.tolist()
            prediction_ls_final = []
            for sublist in prediction_ls:
                for element in sublist:
                    prediction_ls_final.append(element)
            reference_ls += batch_label.tolist()

    print_exams(prediction_ls_final, reference_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
    valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
    valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic
    
    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    ## 将epoch信息写入log.txt
    with open('../../../trained_model/1.3_H3N2_only_model/log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.4f}, valid MAE: {valid_MAE:.4f}, valid MSE: {valid_mse:.4f}, valid Pearson: {valid_pearson:.4f}, valid Spearman: {valid_spearman:.4f}\n")

  1%|          | 2758/441440 [04:08<10:57:52, 11.11it/s]

train loss : 3.7722683030870265


  1%|          | 2760/441440 [04:34<490:11:54,  4.02s/it]

MAE:  1.2325420930793642
MSE:  2.4036036731909283
pearson correlation:  PearsonRResult(statistic=0.5108475981014239, pvalue=1.3618780457418033e-228)
spearman correlation:  SignificanceResult(statistic=0.5179715123051157, pvalue=4.678357753872494e-236)
Validation MSE decrease (inf --> 2.403604).  Saving model ...


  1%|▏         | 5518/441440 [08:43<10:07:47, 11.95it/s] 

...

 90%|█████████ | 397296/441440 [11:00:09<1:01:30, 11.96it/s] 

train loss : 0.8584199719530662


 90%|█████████ | 397296/441440 [11:00:21<1:01:30, 11.96it/s]

MAE:  0.7517794226598764
MSE:  0.9856952023788595
pearson correlation:  PearsonRResult(statistic=0.8349637093139095, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8394572519286859, pvalue=0.0)
EarlyStopping counter: 10 out of 10
Early stopping


: 